In [20]:
import pandas as pd
import re

def parse_gem5_stats_to_dataframe(filepath):
  data = []
  
  stat_pattern = re.compile(r'^\s*([a-zA-Z0-9_\.:-]+)\s+([^#]+?)\s*#\s*(.*)$')
  
  with open(filepath, 'r', encoding='utf-8') as file:
    for line in file:
      line = re.sub(r'\\', '', line).strip()
      
      if not line or line.startswith('---'):
        continue
      
      match = stat_pattern.match(line)
      if match:
        metric = match.group(1).strip()
        raw_values : str = match.group(2).strip()
        description = match.group(3).strip()
        
        tokens = raw_values.split()
        primary_value_str = tokens[0] if tokens else ""
        
        try:
          primary_value = float(primary_value_str)
        except (ValueError, TypeError):
          primary_value = primary_value_str
        
        data.append({
          'Metric': metric,
          'Value': primary_value,
          'Description': description,
          'Raw_Values': raw_values
        })
              
  df = pd.DataFrame(data)
  return df

In [21]:
from matplotlib.axes import Axes
import matplotlib.pyplot as plt
import matplotlib.colors as c
import numpy as np
import pandas as pd
from pathlib import Path
import random

from dataclasses import dataclass

@dataclass
class AlgorithmResult:
    name: str
    dir: Path
    df: pd.DataFrame

@dataclass
class CPUResults:
    name: str
    dir: Path
    results: list[AlgorithmResult]

@dataclass
class Run:
    dir: Path
    cpus: list[CPUResults]

@dataclass
class Metric():
  id: str
  name: str

@dataclass
class Plot():
  name: str
  metrics: list[Metric]
  percentage: bool
  columns_for_total_percentage_sum: list[str]


def maybe_float(s, otherwise: float = 0.0):
  try:
    return float(s)
  except (ValueError, TypeError):
    return otherwise

def get_metric_value(df: pd.DataFrame, id: str):
  row = df[df['Metric'] == id]
  if row.empty: return 0.0

  try:
      return float(row['Value'].values[0])
  except (ValueError, TypeError):
      return 0.0
  
def should_i_be_log(data): # não funciono com negatividade
  return (
      (max_val := np.max(data)) > 0 and
      (min_val := np.min([v for sublist in data for v in sublist if v > 0], initial = 0)) > 0 and
      (max_val / min_val) > 100
  )

def brightness(color):
    # Convert any Matplotlib color format (name, hex, etc.) to 0-1 RGB
    r, g, b = color[:3]
    
    # Standard formula for relative luminance (W3C / ITU-R BT.709)
    # Returns a value between 0.0 (darkest) and 1.0 (brightest)
    return 0.2126 * r + 0.7152 * g + 0.0722 * b

def plot_algorithm_comparison(metrics_to_plot: list[list[Metric]], algorithms: list[AlgorithmResult], name=""):
    num_plots = len(metrics_to_plot)
    
    fig, axes = plt.subplots(num_plots, 1, figsize=(16, 5 * num_plots), sharey=False, squeeze=False)
    axes = axes.flatten()
    fig.suptitle(name)
    cmap = plt.colormaps['hsv']
    colors = [cmap(i) for i in np.linspace(0, 1, len(algorithms))]
    random.shuffle(colors)
    for i, metrics_group in enumerate(metrics_to_plot):
        ax: Axes = axes[i]
        
        x = np.arange(len(metrics_group))
        num_algos = len(algorithms)
        width = 0.8 / num_algos
        
        offsets = [(i - num_algos/2 + 0.5) * width for i in range(num_algos)]
        sort_them=True
        if sort_them:
            
            plot_data = sorted([
                (algo, [maybe_float(get_metric_value(algo.df, metric.id)) for metric in metrics_group], color)
                for algo, color in zip(algorithms, colors)
            ], key=lambda x: x[1])

            for j, (algo, data, color) in enumerate(plot_data):
                b = ax.bar(x + offsets[j], data, width, label=algo.name, color=color)
                # ax.bar_label(b, labels=[f'{v:,.6f}'.rstrip("0").rstrip(".") for v in plot_data[j]], label_type='center', color='white', fontweight='bold')
                ax.bar_label(b, labels=[algo.name for _ in data], label_type='center', color='white' if brightness(color) < 0.5 else 'black', fontweight='bold')
        else:
            
            plot_data = [
                [maybe_float(get_metric_value(algo.df, metric.id)) for metric in metrics_group]
                for algo in algorithms
            ]
            for j, algo in enumerate(algorithms):
                b = ax.bar(x + offsets[j], plot_data[j], width, label=algo.name)
                # ax.bar_label(b, labels=[f'{v:,.6f}'.rstrip("0").rstrip(".") for v in plot_data[j]], label_type='center', color='white', fontweight='bold')
                ax.bar_label(b, labels=[algo.name for _ in plot_data[j]], label_type='center', color=('white'), fontweight='bold')

        # ax.set_title(f"Group {i + 1}", pad=15, fontweight='bold')
        # ax.set_ylabel('Value')
        ax.set_xticks(x)
        
        short_metrics = [m.name for m in metrics_group]
        ax.set_xticklabels(short_metrics, rotation=0)
        
        ax.legend()
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        
        # if should_i_be_log(plot_data):
        #     ax.set_yscale('log')
        #     ax.set_ylabel('Value (Log Scale)')
            
    plt.tight_layout()
    plt.show()
A = AlgorithmResult
M = Metric


In [ ]:
from pathlib import Path

# Define the target directory path
target_dir = Path(".run/run_18")

def newest_run_dir(chud=False, cha=False, radix=False):
    def meets_requirements(d: Path):
        return (
            (not chud or d.joinpath("chud.m5out").exists()) and
            (not cha or d.joinpath("cha.m5out").exists()) and
            (not radix or d.joinpath("radix.m5out").exists())
        )
    return max(
        (d for d in target_dir.iterdir() if d.is_dir() if meets_requirements(d)), 
        key=lambda d: d.stat().st_mtime
    )

def newest_m5out(who: str) -> Path:
    subdir = f"{who}.m5out"
    candidates = [p for d in target_dir.iterdir() if d.is_dir() if (p := d.joinpath(subdir)).exists()]
    if len(candidates) < 1:
        raise ValueError(f'{who} was requested but no "{subdir}" as found')
    return max(candidates, key=lambda d: d.stat().st_mtime)

def subdirs(dir: Path):
    return (d for d in dir.iterdir() if d.is_dir())

def get_run(run_dir: Path) -> Run:
    return Run(run_dir, [
        CPUResults(cpu_dir.name, cpu_dir, [
            AlgorithmResult(algo.name, algo, parse_gem5_stats_to_dataframe(algo.joinpath("stats.txt").as_posix())
            ) for algo in subdirs(cpu_dir) if "orgb_configs" not in algo.name
        ]) for cpu_dir in subdirs(run_dir) if cpu_dir.name.lower().startswith("cpu")
    ])
def get_last_run() -> Run:
    run_dir = max(subdirs(target_dir), key=lambda d: d.stat().st_mtime)
    return get_run(run_dir)
    

run = get_run(target_dir)

In [23]:
metrics_to_plot = [
    # [M('system.cpu.commit.fp_insts', 'float inst')],
    # [M('system.cpu.commit.int_insts', 'int inst')],
    # [M('sim_ticks', 'sim_ticks')],
    [M('system.cpu.dcache.overall_miss_rate::total', 'dcache miss_rate')],
    # [M('system.l2cache.overall_miss_rate::total', 'l2 miss_rate')],
    # [M('system.cpu.ipc', 'ipc')],
    [M('system.cpu.cpi', 'cpi')],
]

# for cpu_result in run.cpus:
#     plot_algorithm_comparison(metrics_to_plot, cpu_result.results, name=cpu_result.name)

unique_algorithms = {algo.name for cpu_result in run.cpus for algo in cpu_result.results}
unique_cpus = {cpu_result.name for cpu_result in run.cpus}
per_algorithm = {algo_name: [AlgorithmResult(cpu.name, algo.dir, algo.df) for cpu in run.cpus for algo in cpu.results if algo.name == algo_name] for algo_name in unique_algorithms}

# i dont work for more than 1 metric per group
metrics = [metrics_group[0] for metrics_group in metrics_to_plot]

metric_scores = {cpu: {metric.name: 0 for metric in metrics} for cpu in unique_cpus}

for name, results in per_algorithm.items():
    plot_data = {
        res.name: {metric.name: maybe_float(get_metric_value(res.df, metric.id)) for metric in metrics}
        for res in results
    }
    for metric in metrics:
        s = sorted([(value, cpu) for cpu, ms in plot_data.items() for m, value in ms.items() if m == metric.name], key=lambda x: x[0])
        for i, (v, cpu) in enumerate(s):
            metric_scores[cpu][metric.name] += i
for metric in metrics:
    s = sorted(metric_scores, key=lambda x: metric_scores[x][metric.name])
    print(f"{metric.name} ->", *[f"({cpu.strip()}: {metric_scores[cpu][metric.name]})" for cpu in s])

for name, results in per_algorithm.items():
    plot_algorithm_comparison(metrics_to_plot, results, name=name)


KeyError: 'Metric'